# Gestire le mail con Python

Gli esempi che usano la rete richiedono un server SMTP/IMAP locale e una
webmail. Questo notebook si basa sull'uso di
[GreenMail](https://greenmail-mail-test.github.io/greenmail/) (server email
in-memory per il testing) e [Roundcube](https://roundcube.net/) (client
webmail), avviabili con `docker compose up` dalla directory corrente.

## Reperire le credenziali

Per prima cosa reperiamo le credenziali per accedere al server SMTP/IMAP e alla
webmail dal file di configurazione `.env` che è usato da `docker-compose.yml`.

In [1]:
import pathlib

# .env contiene: GREENMAIL_USERS=utente:password@dominio,utente2:password2@dominio2
users_str = next(
    l.split('=', 1)[1]
    for l in pathlib.Path('.env').read_text().splitlines()
    if l.startswith('GREENMAIL_USERS=')
)

sender_ap, recipient_ap = users_str.split(',')

def ap(c):  # "utente:password@dominio" → ("utente@dominio", "password")
  up, d = c.split('@')
  u, p = up.split(':')
  return f'{u}@{d}', p

sender, sender_pass = ap(sender_ap)
recipient, recipient_pass = ap(recipient_ap)

## Comporre le email

La libreria standard di Python include un pacchetto
[`email`](https://docs.python.org/3/library/email.html) per comporre, mandare e
leggere le mail (senza coinvolgere la rete).

Ci sono due generazioni di API:

| API | Class | Policy | Notes |
|-----|-------|--------|-------|
| Legacy (< 3.6) | `email.message.Message` | `compat32` | Low-level, gestione dell'encoding fragile |
| **Modern (≥ 3.6)** | **`email.message.EmailMessage`** | **`default` / `EmailPolicy`** | High-level, basata su Unicode, raccomandata |

In [2]:
from email.message import EmailMessage


### Mail semplice

In [3]:
simple_msg = EmailMessage()
simple_msg['Subject'] = 'A simple email'
simple_msg['From'] = sender
simple_msg['To'] = recipient
simple_msg.set_content('This is just some text.')

print(simple_msg)

Subject: A simple email
From: snd@foo.bar
To: rec@goo.bar
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 7bit
MIME-Version: 1.0

This is just some text.



### Con un'alternativa HTML

In [4]:
html_body = """\
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>"""

html_msg = EmailMessage()
html_msg['Subject'] = 'An HTML email'
html_msg['From'] = sender
html_msg['To'] = recipient
html_msg.set_content('Plain text fallback.')
html_msg.add_alternative(html_body, subtype='html')

print(html_msg)

Subject: An HTML email
From: snd@foo.bar
To: rec@goo.bar
MIME-Version: 1.0
Content-Type: multipart/alternative;
 boundary="===============6930717660843794810=="

--===============6930717660843794810==
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 7bit

Plain text fallback.

--===============6930717660843794810==
Content-Type: text/html; charset="utf-8"
Content-Transfer-Encoding: 7bit
MIME-Version: 1.0

<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

--===============6930717660843794810==--



### Con un allegato

In [5]:
csv_data = "name,score\nAlice,95\nBob,87\nCarol,92\n"

attachment_msg = EmailMessage()
attachment_msg['Subject'] = 'A mail with an attachment'
attachment_msg['From'] = sender
attachment_msg['To'] = recipient
attachment_msg.set_content('Please find the results attached.')
attachment_msg.add_attachment(csv_data.encode(), maintype='text', subtype='csv', filename='results.csv')

print(attachment_msg)

Subject: A mail with an attachment
From: snd@foo.bar
To: rec@goo.bar
MIME-Version: 1.0
Content-Type: multipart/mixed; boundary="===============6487865034578303687=="

--===============6487865034578303687==
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 7bit

Please find the results attached.

--===============6487865034578303687==
Content-Type: text/csv
Content-Transfer-Encoding: base64
Content-Disposition: attachment; filename="results.csv"
MIME-Version: 1.0

bmFtZSxzY29yZQpBbGljZSw5NQpCb2IsODcKQ2Fyb2wsOTIK

--===============6487865034578303687==--



### Serializzare e deserializzare i messaggi

In [6]:
import email

# come stringa

raw_str = str(html_msg)

raw_str[:100]


'Subject: An HTML email\nFrom: snd@foo.bar\nTo: rec@goo.bar\nMIME-Version: 1.0\nContent-Type: multipart/a'

In [7]:
# come byte

raw_bytes = html_msg.as_bytes()    # bytes — what actually goes on the wire

raw_bytes[:100]

b'Subject: An HTML email\nFrom: snd@foo.bar\nTo: rec@goo.bar\nMIME-Version: 1.0\nContent-Type: multipart/a'

In [8]:
# Deserializzare 

parsed = email.message_from_bytes(raw_bytes, policy=email.policy.default)

print(f'Content-Type: {parsed.get_content_type()}')
print(f'Subject: {parsed["Subject"]}')


Content-Type: multipart/alternative
Subject: An HTML email


### Esaminare le parti

In [9]:
for part in parsed.walk():
  ct = part.get_content_type()
  cd = part.get_content_disposition()
  print(f'part: {ct}, disposition={cd}')

part: multipart/alternative, disposition=None
part: text/plain, disposition=None
part: text/html, disposition=None


## Inviare le mail con SMTP

Per inviare le mail, usiamo il modulo [`smtplib`](https://docs.python.org/3/library/smtplib.html) della libreria standard. Impostiamo `set_debuglevel(1)` per rendere visibile il dialogo SMTP sottostante — un collegamento diretto con quanto visto a mani nude nel notebook introduttivo.

> **Nota:** `Date` e `Message-ID` sono assenti dal messaggio composto ma saranno
> aggiunti dal server di posta (GreenMail, in conformità con RFC 5321). Per
> aggiungerli lato client si usano `email.utils.formatdate()` e `make_msgid()`.

In [10]:
from smtplib import SMTP

with SMTP('localhost', 3025) as smtp:
    smtp.set_debuglevel(1) # questo consente di ispezionare il protocollo
    smtp.login(sender, sender_pass)
    smtp.send_message(simple_msg)
    smtp.send_message(html_msg)
    smtp.send_message(attachment_msg)

send: 'ehlo [127.0.1.1]\r\n'
reply: b'250-/172.19.0.2\r\n'
reply: b'250 AUTH PLAIN LOGIN XOAUTH2\r\n'
reply: retcode (250); Msg: b'/172.19.0.2\nAUTH PLAIN LOGIN XOAUTH2'
send: 'AUTH PLAIN AHNuZEBmb28uYmFyAHNuZHA=\r\n'
reply: b'235 2.7.0  Authentication Succeeded\r\n'
reply: retcode (235); Msg: b'2.7.0  Authentication Succeeded'
send: 'mail FROM:<snd@foo.bar>\r\n'
reply: b'250 OK\r\n'
reply: retcode (250); Msg: b'OK'
send: 'rcpt TO:<rec@goo.bar>\r\n'
reply: b'250 OK\r\n'
reply: retcode (250); Msg: b'OK'
send: 'data\r\n'
reply: b'354 Start mail input; end with <CRLF>.<CRLF>\r\n'
reply: retcode (354); Msg: b'Start mail input; end with <CRLF>.<CRLF>'
data: (354, b'Start mail input; end with <CRLF>.<CRLF>')
send: b'Subject: A simple email\r\nFrom: snd@foo.bar\r\nTo: rec@goo.bar\r\nContent-Type: text/plain; charset="utf-8"\r\nContent-Transfer-Encoding: 7bit\r\nMIME-Version: 1.0\r\n\r\nThis is just some text.\r\n.\r\n'
reply: b'250 OK\r\n'
reply: retcode (250); Msg: b'OK'
data: (250, b'OK')
s

## Ricevere la mail con IMAP

### Usando la libreria standard

In [11]:
import imaplib, email as emaillib

with imaplib.IMAP4('localhost', 3143) as imap:
  imap.debug = 4 # questo consente di ispezionare il protocollo
  imap.login(recipient, recipient_pass)
  imap.select('INBOX')
  _, msg_nums = imap.search(None, 'ALL')
  for num in msg_nums[0].split():
    _, data = imap.fetch(num, '(RFC822)')
    msg = emaillib.message_from_bytes(data[0][1], policy=emaillib.policy.default)
    print(f"Subject: {msg['Subject']}")
    for part in msg.walk():
      ct = part.get_content_type()
      cd = part.get_content_disposition()
      if cd == 'attachment':
        print(f'  attachment: {part.get_filename()}')
        print(part.get_payload(decode=True).decode())
      elif ct == 'text/html':
        print('  html alternative:')
        print(part.get_payload(decode=True).decode())

  22:23.41 > b'DLMC1 LOGIN rec@goo.bar "recp"'
  22:23.42 < b'DLMC1 OK LOGIN completed.'
  22:23.42 > b'DLMC2 SELECT INBOX'
  22:23.42 < b'* FLAGS (\\Answered \\Deleted \\Draft \\Flagged \\Seen)'
  22:23.46 < b'* 24 EXISTS'
  22:23.46 < b'* 3 RECENT'
  22:23.46 < b'* OK [UIDVALIDITY 1780659591]'
  22:23.46 < b'* OK [UIDNEXT 25]'
  22:23.46 < b'* OK [UNSEEN 22] Message 22 is the first unseen'
  22:23.46 < b'* OK [PERMANENTFLAGS (\\Answered \\Deleted \\Draft \\Flagged \\Seen \\*)]'
  22:23.46 < b'DLMC2 OK [READ-WRITE] SELECT completed.'
  22:23.47 > b'DLMC3 SEARCH ALL'
  22:23.47 < b'* SEARCH 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24'
  22:23.51 < b'DLMC3 OK SEARCH completed.'
  22:23.51 > b'DLMC4 FETCH 1 (RFC822)'
  22:23.51 < b'* 1 FETCH (RFC822 {285}'
  22:23.51 read literal size 285
  22:23.51 < b')'
  22:23.55 < b'DLMC4 OK FETCH completed.'
  22:23.55 > b'DLMC5 FETCH 2 (RFC822)'
  22:23.56 < b'* 2 FETCH (RFC822 {636}'
  22:23.56 read literal size 636
  22:23.56 

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body><h2>Hello!</h2></body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>



  22:23.77 < b'DLMC9 OK FETCH completed.'
  22:23.77 > b'DLMC10 FETCH 7 (RFC822)'
  22:23.77 < b'* 7 FETCH (RFC822 {285}'
  22:23.77 read literal size 285
  22:23.77 < b')'
  22:23.82 < b'DLMC10 OK FETCH completed.'
  22:23.82 > b'DLMC11 FETCH 8 (RFC822)'
  22:23.82 < b'* 8 FETCH (RFC822 {700}'
  22:23.82 read literal size 700
  22:23.82 < b')'
  22:23.86 < b'DLMC11 OK FETCH completed.'
  22:23.86 > b'DLMC12 FETCH 9 (RFC822)'
  22:23.86 < b'* 9 FETCH (RFC822 {701}'
  22:23.86 read literal size 701
  22:23.86 < b')'
  22:23.90 < b'DLMC12 OK FETCH completed.'
  22:23.90 > b'DLMC13 FETCH 10 (RFC822)'
  22:23.91 < b'* 10 FETCH (RFC822 {285}'
  22:23.91 read literal size 285
  22:23.91 < b')'
  22:23.95 < b'DLMC13 OK FETCH completed.'
  22:23.95 > b'DLMC14 FETCH 11 (RFC822)'
  22:23.95 < b'* 11 FETCH (RFC822 {700}'
  22:23.95 read literal size 700
  22:23.95 < b')'


Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email


  22:23.99 < b'DLMC14 OK FETCH completed.'
  22:23.99 > b'DLMC15 FETCH 12 (RFC822)'
  22:23.99 < b'* 12 FETCH (RFC822 {701}'
  22:23.99 read literal size 701
  22:23.99 < b')'
  22:24.03 < b'DLMC15 OK FETCH completed.'
  22:24.04 > b'DLMC16 FETCH 13 (RFC822)'
  22:24.04 < b'* 13 FETCH (RFC822 {285}'
  22:24.04 read literal size 285
  22:24.04 < b')'
  22:24.08 < b'DLMC16 OK FETCH completed.'
  22:24.08 > b'DLMC17 FETCH 14 (RFC822)'
  22:24.08 < b'* 14 FETCH (RFC822 {700}'
  22:24.08 read literal size 700
  22:24.08 < b')'
  22:24.12 < b'DLMC17 OK FETCH completed.'
  22:24.12 > b'DLMC18 FETCH 15 (RFC822)'
  22:24.12 < b'* 15 FETCH (RFC822 {701}'
  22:24.12 read literal size 701
  22:24.12 < b')'
  22:24.16 < b'DLMC18 OK FETCH completed.'
  22:24.17 > b'DLMC19 FETCH 16 (RFC822)'
  22:24.17 < b'* 16 FETCH (RFC822 {285}'
  22:24.17 read literal size 285
  22:24.17 < b')'


Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92



  22:24.21 < b'DLMC19 OK FETCH completed.'
  22:24.21 > b'DLMC20 FETCH 17 (RFC822)'
  22:24.21 < b'* 17 FETCH (RFC822 {700}'
  22:24.21 read literal size 700
  22:24.21 < b')'
  22:24.25 < b'DLMC20 OK FETCH completed.'
  22:24.25 > b'DLMC21 FETCH 18 (RFC822)'
  22:24.25 < b'* 18 FETCH (RFC822 {701}'
  22:24.25 read literal size 701
  22:24.26 < b')'
  22:24.30 < b'DLMC21 OK FETCH completed.'
  22:24.30 > b'DLMC22 FETCH 19 (RFC822)'
  22:24.30 < b'* 19 FETCH (RFC822 {285}'
  22:24.30 read literal size 285
  22:24.30 < b')'
  22:24.34 < b'DLMC22 OK FETCH completed.'
  22:24.34 > b'DLMC23 FETCH 20 (RFC822)'
  22:24.34 < b'* 20 FETCH (RFC822 {700}'
  22:24.34 read literal size 700
  22:24.34 < b')'
  22:24.38 < b'DLMC23 OK FETCH completed.'
  22:24.38 > b'DLMC24 FETCH 21 (RFC822)'
  22:24.39 < b'* 21 FETCH (RFC822 {701}'
  22:24.39 read literal size 701
  22:24.39 < b')'


Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>



  22:24.43 < b'DLMC24 OK FETCH completed.'
  22:24.43 > b'DLMC25 FETCH 22 (RFC822)'
  22:24.43 < b'* 22 FETCH (FLAGS (\\Seen) RFC822 {285}'
  22:24.43 read literal size 285
  22:24.43 < b')'
  22:24.47 < b'DLMC25 OK FETCH completed.'
  22:24.47 > b'DLMC26 FETCH 23 (RFC822)'
  22:24.47 < b'* 23 FETCH (FLAGS (\\Seen) RFC822 {700}'
  22:24.47 read literal size 700
  22:24.47 < b')'
  22:24.51 < b'DLMC26 OK FETCH completed.'
  22:24.52 > b'DLMC27 FETCH 24 (RFC822)'
  22:24.52 < b'* 24 FETCH (FLAGS (\\Seen) RFC822 {701}'
  22:24.52 read literal size 701
  22:24.52 < b')'
  22:24.56 < b'DLMC27 OK FETCH completed.'
  22:24.56 > b'DLMC28 LOGOUT'
  22:24.56 < b'* BYE IMAP4rev1 Server logging out'
  22:24.56 BYE response: b'IMAP4rev1 Server logging out'


Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92



### Usando imap-tools

`imaplib` è completa ma di basso livello: le risposte sono byte grezzi, i criteri di ricerca sono stringhe non tipizzate e tutto va analizzato manualmente. La libreria di terze parti [`imap-tools`](https://github.com/ikvk/imap_tools) la incapsula con un'API pulita e pythonica.

| Operazione | `imaplib` | `imap-tools` |
|------------|-----------|--------------|
| Recupera tutti i messaggi | `search` + ciclo `fetch` | iteratore `mailbox.fetch()` |
| Legge il subject | `email.message_from_bytes(raw)['Subject']` | `msg.subject` |
| Elenca gli allegati | scorrere le parti, filtrare per disposition | `msg.attachments` |
| Filtra lato server | criteri in stringa | `AND(from_='alice@…', seen=False)` |

In [12]:
from imap_tools import MailBoxUnencrypted, AND

with MailBoxUnencrypted('localhost', 3143).login(recipient, recipient_pass) as mailbox:
  for msg in mailbox.fetch(AND(from_='snd@foo.bar')):
    print(f'Subject: {msg.subject}')
    for att in msg.attachments:
      print(f'  attachment: {att.filename}')
      print(att.payload.decode())
    if msg.html:
      print('  html alternative:')
      print(msg.html)


Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body><h2>Hello!</h2></body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

S